# Testing Framework 
Current Goal: 
-Simple Qbit input (QC object)
-Take expected output
-import Asim's algorithm and use it  
-take in shot count 
-Use AER simulator

In [8]:
#imports 
import os

from dotenv import load_dotenv
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

# potential addition 
using regex to try to make it easier to determing which simulator or computer the user wants to use even with mispellings 

In [9]:
import os
from dotenv import load_dotenv
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService


def get_backend(backend_name):
    """
    @brief Validates and retrieves a quantum computing backend.

    @param backend_name Name of the requested backend.
    @return Dictionary containing backend object, type, and name.
    """

    # Validate backend name
    if not isinstance(backend_name, str):
        raise TypeError("Backend name must be a string.")

    backend_name = backend_name.strip()

    if not backend_name:
        raise ValueError("Backend name cannot be empty.")

    # Local simulator
    if backend_name == "aer_simulator":
        return {
            "backend": AerSimulator(),
            "type": "simulator",
            "name": backend_name
        }

    # Confirm quantum computer selection before accessing credentials
    confirmation = input(
        f"You selected to use the quantum computer '{backend_name}'. "
        "Are you sure you want to continue? (Y/N): "
    ).strip().upper()

    if confirmation != "Y":
        raise PermissionError(
            "Quantum computer selection cancelled."
        )

    # Load IBM Quantum credentials
    load_dotenv()

    token = os.getenv("IBM_QUANTUM_API_KEY")
    instance = os.getenv("IBM_QUANTUM_INSTANCE")

    if not token or not instance:
        raise ValueError(
            "IBM Quantum credentials are missing."
        )

    # Connect to IBM Quantum and validate backend
    try:
        service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=token,
            instance=instance
        )

        backend = service.backend(backend_name)

    except Exception as error:
        raise ValueError(
            f"Unable to access backend '{backend_name}': {error}"
        ) from error

    # Verify selected backend is a real quantum computer
    if backend.configuration().simulator:
        raise ValueError(
            "Selected backend is a simulator, not a real quantum computer."
        )

    # Check backend availability
    if not backend.status().operational:
        raise RuntimeError(
            f"Backend '{backend_name}' is not operational."
        )

    # Return validated quantum computer
    return {
        "backend": backend,
        "type": "hardware",
        "name": backend_name
    }

In [10]:
def testing_framework(
    qbits,
    expected_output,
    backend="aer_simulator",
    shots=1000,
    accuracy=0.90
):
    """
    @brief Tests a quantum circuit on a selected backend.

    @param qbits Qiskit QuantumCircuit object.
    @param expected_output Expected measurement bitstring.
    @param backend Name of the selected backend.
    @param shots Number of circuit executions.
    @param accuracy Required success rate (0.0 to 1.0).

    @return Dictionary of measurement results and pass/fail status.
    """

    # Validate circuit
    if not isinstance(qbits, QuantumCircuit):
        raise TypeError("qbits must be a QuantumCircuit.")

    if qbits.num_qubits == 0:
        raise ValueError("Circuit must contain at least one qubit.")

    if qbits.num_clbits != 0:
        raise ValueError("Circuit must not contain classical bits.")

    if qbits.parameters:
        raise ValueError("Circuit contains unbound parameters.")

    # Validate expected output
    if not isinstance(expected_output, str):
        raise TypeError("Expected output must be a string.")

    if len(expected_output) != qbits.num_qubits:
        raise ValueError("Expected output length must match qubits.")

    if any(bit not in "01" for bit in expected_output):
        raise ValueError("Expected output must contain only 0 and 1.")

    # Validate shots
    if type(shots) is not int or shots <= 0:
        raise ValueError("Shots must be a positive integer.")

    # Validate accuracy
    if type(accuracy) not in (int, float):
        raise TypeError("Accuracy must be numeric.")

    if not 0 <= accuracy <= 1:
        raise ValueError("Accuracy must be between 0 and 1.")

    # Retrieve and validate backend
    backend_info = get_backend(backend)

    selected_backend = backend_info["backend"]
    backend_type = backend_info["type"]

    # Block hardware execution until implemented
    if backend_type == "hardware":
        raise NotImplementedError(
            "Real quantum computer execution is not implemented yet."
        )

    # Prepare circuit
    circuit = qbits.measure_all(inplace=False)
    compiled_circuit = transpile(circuit, selected_backend)

    # Execute circuit
    job = selected_backend.run(compiled_circuit, shots=shots)
    result = job.result()

    # Retrieve counts
    counts = result.get_counts()

    # Calculate success rate
    successful_shots = counts.get(expected_output, 0)
    success_rate = successful_shots / shots

    # Determine pass/fail
    passed = success_rate >= accuracy

    # Return results
    return {
        "backend": backend_info["name"],
        "backend_type": backend_type,
        "qubits": qbits.num_qubits,
        "expected_output": expected_output,
        "shots": shots,
        "counts": counts,
        "successful_shots": successful_shots,
        "success_rate": success_rate,
        "required_accuracy": accuracy,
        "passed": passed
    }